# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [3]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\juani\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\juani\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\juani\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Descargar recursos de nltk
nltk.download('punkt')
nltk.download('stopwords')

# =========================
# Cargar el corpus
# =========================

ruta = r"C:\Users\juani\OneDrive\Escritorio\wikipedia_text_corpus.csv"

df = pd.read_csv(ruta)

# Mostrar información básica
print(df.head())
print(df.columns)

# =========================
# Seleccionar la columna de texto
# =========================

# Cambia "text" si tu columna tiene otro nombre
textos = df['text'].astype(str)

# =========================
# Preprocesamiento
# =========================

stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Minúsculas
    text = text.lower()
    
    # Eliminar caracteres especiales y números
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenización
    tokens = word_tokenize(text)
    
    # Eliminar stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

# Aplicar preprocesamiento
df['processed_text'] = textos.apply(preprocess)

print(df[['processed_text']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\juani\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\juani\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


   Unnamed: 0                                               text
0           1  Anovo\n\nAnovo (formerly A Novo) is a computer...
1           2  Battery indicator\n\nA battery indicator (also...
2           3  Bob Pease\n\nRobert Allen Pease (August 22, 19...
3           4  CAVNET\n\nCAVNET was a secure military forum w...
4           5  CLidar\n\nThe CLidar is a scientific instrumen...
Index(['Unnamed: 0', 'text'], dtype='object')
                                      processed_text
0  anovo anovo formerly novo computer services co...
1  battery indicator battery indicator also known...
2  bob pease robert allen pease august june analo...
3  cavnet cavnet secure military forum became ope...
4  clidar clidar scientific instrument used measu...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# Vectorización TF-IDF
# =========================

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df['processed_text'])

print("Dimensiones matriz TF-IDF:", tfidf_matrix.shape)

# =========================
# Queries de prueba
# =========================

queries = [
    "artificial intelligence",
    "machine learning",
    "computer science",
    "history of europe",
    "space exploration",
    "climate change",
    "human biology",
    "world war",
    "internet technology",
    "quantum physics"
]

# =========================
# Recuperación
# =========================

for query in queries:
    query_processed = preprocess(query)
    
    query_vector = vectorizer.transform([query_processed])
    
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = similarities.argsort()[-5:][::-1]
    
    print("\nQUERY:", query)
    print("-" * 50)
    
    for idx in top_indices:
        print(f"Documento {idx} | Score: {similarities[idx]:.4f}")
        print(df['text'].iloc[idx][:200])
        print()

Dimensiones matriz TF-IDF: (10859, 173934)

QUERY: artificial intelligence
--------------------------------------------------
Documento 5229 | Score: 0.5861
Artificial psychology

Artificial psychology is a theoretical discipline proposed by Dan Curtis (b. 1963). The theory considers the situation when an artificial intelligence approaches the level of co

Documento 3905 | Score: 0.4594
Accounting intelligence

A specialist form of business intelligence, accounting intelligence is the general name for the set of technologies used to extract, analyse and present information from accou

Documento 4563 | Score: 0.4455
Intelligence engine

An intelligence engine is a type of enterprise information management that combines business rule management, predictive, and prescriptive analytics to form a unified information-

Documento 8176 | Score: 0.3630
Robert Trappl

Robert Trappl (born 16 January 1939 in Vienna) is an Austrian scientist and head of the Austrian Research Institute for Artificia

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [6]:
from rank_bm25 import BM25Okapi

# =========================
# Tokenización del corpus
# =========================

tokenized_corpus = [
    doc.split(" ")
    for doc in df['processed_text']
]

# Crear modelo BM25
bm25 = BM25Okapi(tokenized_corpus)

# =========================
# Recuperación BM25
# =========================

for query in queries:
    tokenized_query = preprocess(query).split(" ")
    
    scores = bm25.get_scores(tokenized_query)
    
    top_indices = np.argsort(scores)[-5:][::-1]
    
    print("\nQUERY:", query)
    print("-" * 50)
    
    for idx in top_indices:
        print(f"Documento {idx} | Score: {scores[idx]:.4f}")
        print(df['text'].iloc[idx][:200])
        print()


QUERY: artificial intelligence
--------------------------------------------------
Documento 5229 | Score: 15.0999
Artificial psychology

Artificial psychology is a theoretical discipline proposed by Dan Curtis (b. 1963). The theory considers the situation when an artificial intelligence approaches the level of co

Documento 8176 | Score: 14.5018
Robert Trappl

Robert Trappl (born 16 January 1939 in Vienna) is an Austrian scientist and head of the Austrian Research Institute for Artificial Intelligence in Vienna, which was founded in 1984. He 

Documento 669 | Score: 14.2248
Open Letter on Artificial Intelligence

In January 2015, Stephen Hawking, Elon Musk, and dozens of artificial intelligence experts signed an open letter on artificial intelligence calling for research

Documento 485 | Score: 13.8076
Strategic Computing Initiative

The United States government's Strategic Computing Initiative funded research into advanced computer hardware and artificial intelligence from 1983 to 19

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [7]:
print("COMPARACIÓN TF-IDF vs BM25")
print("=" * 60)

for query in queries:
    
    print(f"\nQUERY: {query}")
    
    # TF-IDF
    query_vector = vectorizer.transform([preprocess(query)])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    tfidf_top = similarities.argsort()[-3:][::-1]
    
    # BM25
    bm25_scores = bm25.get_scores(preprocess(query).split())
    bm25_top = np.argsort(bm25_scores)[-3:][::-1]
    
    print("\nTop TF-IDF:")
    for idx in tfidf_top:
        print(f"Doc {idx}")
    
    print("\nTop BM25:")
    for idx in bm25_top:
        print(f"Doc {idx}")
    
    print("-" * 60)

COMPARACIÓN TF-IDF vs BM25

QUERY: artificial intelligence

Top TF-IDF:
Doc 5229
Doc 3905
Doc 4563

Top BM25:
Doc 5229
Doc 8176
Doc 669
------------------------------------------------------------

QUERY: machine learning

Top TF-IDF:
Doc 10547
Doc 7900
Doc 6853

Top BM25:
Doc 10547
Doc 4733
Doc 10699
------------------------------------------------------------

QUERY: computer science

Top TF-IDF:
Doc 9489
Doc 9852
Doc 5814

Top BM25:
Doc 9489
Doc 9852
Doc 10579
------------------------------------------------------------

QUERY: history of europe

Top TF-IDF:
Doc 6079
Doc 1600
Doc 3981

Top BM25:
Doc 504
Doc 3958
Doc 10331
------------------------------------------------------------

QUERY: space exploration

Top TF-IDF:
Doc 4083
Doc 7218
Doc 1066

Top BM25:
Doc 4083
Doc 7218
Doc 1066
------------------------------------------------------------

QUERY: climate change

Top TF-IDF:
Doc 10805
Doc 941
Doc 2547

Top BM25:
Doc 4823
Doc 10805
Doc 954
----------------------------------------